In [1]:
!pip -q install sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.1 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [3]:
faq_data = [
    {
        "question": "What are the clinic timings?",
        "answer": "The clinic is open from 9:00 AM to 8:00 PM Monday to Saturday."
    },
    {
        "question": "Is the clinic open on Sunday?",
        "answer": "The clinic is closed on Sunday."
    },
    {
        "question": "How can I book an appointment?",
        "answer": "Appointments can be booked through the clinic reception or by phone."
    },
    {
        "question": "Does the clinic provide emergency services?",
        "answer": "The clinic does not provide 24-hour emergency services. Please contact an emergency hospital for emergencies."
    },
    {
        "question": "What services are available?",
        "answer": "The clinic provides general consultation, health checkups, laboratory tests, vaccination, and specialist consultation."
    },
    {
        "question": "Where is the clinic located?",
        "answer": "The clinic is located at 25 Anna Nagar Main Road, Chennai, Tamil Nadu."
    },
    {
        "question": "Can I cancel my appointment?",
        "answer": "Yes, appointments can be cancelled by contacting the clinic reception."
    },
    {
        "question": "Do I need an appointment?",
        "answer": "Appointments are recommended, but walk-in consultations may be available depending on doctor availability."
    },
    {
        "question": "What payment methods are accepted?",
        "answer": "The clinic accepts cash, UPI, debit cards, and credit cards."
    },
    {
        "question": "Does the clinic provide vaccination?",
        "answer": "Yes, vaccination services are available at the clinic."
    }
]

documents = [
    item["question"] + " " + item["answer"]
    for item in faq_data
]

print("Number of FAQ documents:", len(documents))

Number of FAQ documents: 10


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    documents,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (10, 384)


In [5]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print("Documents stored in vector database:", index.ntotal)

Documents stored in vector database: 10


In [7]:
def retrieve_documents(query, k=3):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k
    )

    results = []

    for i in indices[0]:
        results.append(documents[i])

    return results

In [8]:
def rag_chatbot(query):

    retrieved_docs = retrieve_documents(query, k=3)

    print("Retrieved Information:")
    print("-" * 50)

    for doc in retrieved_docs:
        print(doc)

    print("\nChatbot Answer:")
    print("-" * 50)

    # Select the most relevant document
    answer = retrieved_docs[0]

    # Extract answer from the FAQ
    for item in faq_data:
        combined = item["question"] + " " + item["answer"]

        if combined == answer:
            return item["answer"]

    return answer

In [9]:
question = "What time does the clinic open?"

answer = rag_chatbot(question)

print(answer)

Retrieved Information:
--------------------------------------------------
What are the clinic timings? The clinic is open from 9:00 AM to 8:00 PM Monday to Saturday.
Is the clinic open on Sunday? The clinic is closed on Sunday.
How can I book an appointment? Appointments can be booked through the clinic reception or by phone.

Chatbot Answer:
--------------------------------------------------
The clinic is open from 9:00 AM to 8:00 PM Monday to Saturday.


In [10]:
questions = [
    "When is the clinic open?",
    "How do I make an appointment?",
    "Where is the clinic?",
    "Can I pay using UPI?",
    "Does the clinic give vaccinations?"
]

for question in questions:

    print("\nUser:", question)

    answer = rag_chatbot(question)

    print("Bot:", answer)


User: When is the clinic open?
Retrieved Information:
--------------------------------------------------
What are the clinic timings? The clinic is open from 9:00 AM to 8:00 PM Monday to Saturday.
Is the clinic open on Sunday? The clinic is closed on Sunday.
How can I book an appointment? Appointments can be booked through the clinic reception or by phone.

Chatbot Answer:
--------------------------------------------------
Bot: The clinic is open from 9:00 AM to 8:00 PM Monday to Saturday.

User: How do I make an appointment?
Retrieved Information:
--------------------------------------------------
How can I book an appointment? Appointments can be booked through the clinic reception or by phone.
Do I need an appointment? Appointments are recommended, but walk-in consultations may be available depending on doctor availability.
Can I cancel my appointment? Yes, appointments can be cancelled by contacting the clinic reception.

Chatbot Answer:
-------------------------------------------

In [ ]:
print("🏥 Clinic FAQ RAG Chatbot")
print("Type 'exit' to stop the chatbot.")
print("=" * 50)

while True:

    user_question = input("\nYou: ")

    if user_question.lower() == "exit":
        print("Bot: Thank you for using the Clinic FAQ Chatbot!")
        break

    answer = rag_chatbot(user_question)

    print("Bot:", answer)

🏥 Clinic FAQ RAG Chatbot
Type 'exit' to stop the chatbot.
Retrieved Information:
--------------------------------------------------
What are the clinic timings? The clinic is open from 9:00 AM to 8:00 PM Monday to Saturday.
How can I book an appointment? Appointments can be booked through the clinic reception or by phone.
What services are available? The clinic provides general consultation, health checkups, laboratory tests, vaccination, and specialist consultation.

Chatbot Answer:
--------------------------------------------------
Bot: The clinic is open from 9:00 AM to 8:00 PM Monday to Saturday.
Retrieved Information:
--------------------------------------------------
What payment methods are accepted? The clinic accepts cash, UPI, debit cards, and credit cards.
Can I cancel my appointment? Yes, appointments can be cancelled by contacting the clinic reception.
Does the clinic provide emergency services? The clinic does not provide 24-hour emergency services. Please contact an emer